# Week 14: Mini Project — Sensor Log Summary — PHASE 7: Proving Mastery
### *Core Mastery: "I can solve a problem step by step with code"*

*📚 Computer Programming I · ⏱️ 5 Hours · 👨‍🏫 Dr. Arif Solmaz*

## 🎯 Learning Objectives

By the end of this week, you will be able to:

- Apply file I/O and CSV skills in a real-world project
- Read and parse CSV sensor data from a file
- Validate and clean messy data with error handling
- Compute statistics (min, max, mean, count) per sensor
- Generate formatted text reports
- Write cleaned data and summaries to output files
- Combine multiple functions into a complete program

## 🎯 Core Mastery Connection

**Core Mastery:** *"I can solve a problem step by step with code"* — Algorithmic Thinking

This is the proof. You take a real problem, decompose it into steps, and build a complete working solution using everything you have learned. This mini project is not about learning a new concept — it is about demonstrating that you can independently break down a multi-step problem (read, clean, analyze, report) and express each step as working Python code.

---
## Part 1: Project Overview

In this project, you will build a **Sensor Log Summary Tool**. Imagine you work at a weather station that collects data from multiple sensors (temperature, humidity, pressure). The data is stored in a CSV file, but some readings are **corrupted or missing**.

Your tool will:
1. **Read** sensor data from a CSV file
2. **Clean** the data by removing bad rows
3. **Compute statistics** for each sensor (min, max, mean, count)
4. **Generate** a formatted summary report
5. **Write** cleaned data to a new CSV file
6. **Write** the summary report to a text file

This project brings together everything you have learned in this course: variables, data types, conditionals, loops, functions, lists, strings, and file I/O.

> 💡 **Note:** Work through each step in order. Each exercise builds on the previous one. By the end, you will combine all steps into one complete program.

---
## Part 2: Project Requirements

### Input
A CSV file called `sensor_data.csv` with three columns:

| Column | Description | Example |
|:---|:---|:---|
| `timestamp` | Date and time of reading | `2024-01-15 08:00` |
| `sensor` | Name of the sensor | `temp`, `humidity`, `pressure` |
| `value` | The sensor reading | `22.5`, `45.2`, `1013.25` |

### Bad Data (to be cleaned)
The file may contain:
- **Empty values** — the value field is blank
- **Non-numeric values** — the value field contains text like `abc`
- **Impossible values** — values that don't make physical sense (e.g., negative humidity)

### Validation Rules

| Sensor | Valid Range |
|:---|:---|
| `temp` | -50.0 to 60.0 °C |
| `humidity` | 0.0 to 100.0 % |
| `pressure` | 800.0 to 1200.0 hPa |

### Output
1. `sensor_data_clean.csv` — cleaned data (bad rows removed)
2. `sensor_report.txt` — formatted summary report with statistics

---
## Part 3: Step 1 — Create Sample Data

Let's create our sensor data file with some intentionally bad data. Run the cell below to create the file.

**Figure 3.1: Creating the sample sensor data CSV file**

In [ ]:
# ✏️ [EX1]
# Run this cell to create the sample data file.
# You do NOT need to modify this cell.

sample_data = """timestamp,sensor,value
2024-01-15 08:00,temp,22.5
2024-01-15 08:05,humidity,45.2
2024-01-15 08:10,temp,23.1
2024-01-15 08:15,pressure,1013.25
2024-01-15 08:20,temp,-999
2024-01-15 08:25,humidity,
2024-01-15 08:30,temp,22.8
2024-01-15 08:35,humidity,46.1
2024-01-15 08:40,pressure,abc
2024-01-15 08:45,temp,23.5
2024-01-15 08:50,humidity,44.8
2024-01-15 08:55,pressure,1012.80
2024-01-15 09:00,temp,24.0
2024-01-15 09:05,humidity,150.0
2024-01-15 09:10,pressure,1013.50
2024-01-15 09:15,temp,22.2
2024-01-15 09:20,humidity,43.5
2024-01-15 09:25,pressure,1011.90
2024-01-15 09:30,temp,
2024-01-15 09:35,humidity,47.3"""

with open("sensor_data.csv", "w") as f:
    f.write(sample_data)

print("\u2705 sensor_data.csv created successfully!")
print(f"File size: {len(sample_data)} bytes")

# Show the file contents
print("\n--- File Contents ---")
with open("sensor_data.csv", "r") as f:
    for i, line in enumerate(f):
        marker = ""
        # Mark potentially bad rows
        stripped = line.strip()
        if stripped.endswith(",") or ",," in stripped:
            marker = "  <-- EMPTY VALUE"
        elif "-999" in stripped:
            marker = "  <-- SUSPICIOUS VALUE"
        elif "abc" in stripped:
            marker = "  <-- NON-NUMERIC"
        elif "150.0" in stripped:
            marker = "  <-- OUT OF RANGE"
        print(f"  {i}: {stripped}{marker}")

---
## Part 4: Step 2 — Read the Data

Now let's read the CSV file and parse it into a list of lists. Each inner list should contain `[timestamp, sensor, value_string]`.

**Figure 4.1: Example of reading and parsing CSV data**

In [ ]:
# Example: How to read a CSV into a list of lists
# (This is a demonstration - you'll write your own version below)

with open("sensor_data.csv", "r") as f:
    header = f.readline().strip().split(",")
    print("Header:", header)
    
    # Read just the first 3 data rows as example
    for i in range(3):
        line = f.readline().strip()
        row = line.split(",")
        print(f"Row {i+1}: {row}")

### Exercise 2: Read and Parse the CSV

Write code that reads `"sensor_data.csv"` and stores all data rows in a list called `raw_data`. Each element should be a list of 3 strings: `[timestamp, sensor, value]`.

**Expected output:**
```
Header: ['timestamp', 'sensor', 'value']
Total rows read: 20
First row: ['2024-01-15 08:00', 'temp', '22.5']
Last row: ['2024-01-15 09:35', 'humidity', '47.3']
```

<details><summary>💡 Hint</summary>

Read the first line as the header. Then loop through the remaining lines, `strip()` each line, `split(",")` it, and `append()` to `raw_data`.
</details>

In [ ]:
# ✏️ [EX2]


---
## Part 5: Step 3 — Clean the Data

Now we need to validate each row and separate good data from bad data. A row is **bad** if:
- The value field is **empty**
- The value is **not a valid number**
- The value is **outside the valid range** for that sensor

### Validation Ranges Reminder

| Sensor | Min | Max |
|:---|:---|:---|
| `temp` | -50.0 | 60.0 |
| `humidity` | 0.0 | 100.0 |
| `pressure` | 800.0 | 1200.0 |

### Exercise 3: Validate a Single Row

Write a function called `validate_row(row)` that takes a list `[timestamp, sensor, value_string]` and returns a tuple `(is_valid, reason)`.

- If the row is valid: return `(True, "OK")`
- If the value is empty: return `(False, "empty value")`
- If the value is not a number: return `(False, "non-numeric value: 'xxx'")`
- If the value is out of range: return `(False, "out of range: xxx")`

**Expected output (test cases):**
```
['2024-01-15 08:00', 'temp', '22.5']     -> (True, 'OK')
['2024-01-15 08:25', 'humidity', '']      -> (False, 'empty value')
['2024-01-15 08:40', 'pressure', 'abc']   -> (False, "non-numeric value: 'abc'")
['2024-01-15 08:20', 'temp', '-999']      -> (False, 'out of range: -999.0')
['2024-01-15 09:05', 'humidity', '150.0'] -> (False, 'out of range: 150.0')
```

<details><summary>💡 Hint</summary>

1. Check if `row[2].strip()` is empty.
2. Try `float(row[2])` in a `try/except ValueError`.
3. Use a dictionary for ranges: `{"temp": (-50, 60), "humidity": (0, 100), "pressure": (800, 1200)}`.
4. Check if the value is within the range for the given sensor.
</details>

In [ ]:
# ✏️ [EX3]


### Exercise 4: Clean the Data

Using your `validate_row()` function, separate the data into two lists:
- `clean_data` — rows that passed validation
- `bad_data` — rows that failed, along with the reason

Print a summary of the cleaning results.

**Expected output:**
```
Data Cleaning Results
=====================
Total rows: 20
Clean rows: 15
Bad rows: 5

Bad rows detail:
  Row 5: 2024-01-15 08:20 | temp | -999 -> out of range: -999.0
  Row 6: 2024-01-15 08:25 | humidity |  -> empty value
  Row 9: 2024-01-15 08:40 | pressure | abc -> non-numeric value: 'abc'
  Row 14: 2024-01-15 09:05 | humidity | 150.0 -> out of range: 150.0
  Row 19: 2024-01-15 09:30 | temp |  -> empty value
```

<details><summary>💡 Hint</summary>

Loop through `raw_data` with `enumerate()` to track row numbers. Call `validate_row()` on each row. Append valid rows to `clean_data` and invalid ones (with reason) to `bad_data`.
</details>

In [ ]:
# ✏️ [EX4]


---
## Part 6: Step 4 — Compute Statistics

Now let's compute statistics for each sensor type. We need to calculate:
- **Count** — how many valid readings
- **Min** — smallest value
- **Max** — largest value
- **Mean** — average value

### Exercise 5: Stats for One Sensor

Write a function called `compute_stats(data, sensor_name)` that takes the clean data list and a sensor name, and returns a dictionary with `count`, `min`, `max`, and `mean` for that sensor.

**Expected output (test):**
```
Stats for temp:
  count: 5
  min: 22.2
  max: 24.0
  mean: 23.12
```

<details><summary>💡 Hint</summary>

1. Filter `clean_data` to only include rows where `row[1] == sensor_name`.
2. Extract the values with `float(row[2])`.
3. Use `len()`, `min()`, `max()`, and `sum()/len()` on the values list.
4. Return a dictionary: `{"count": ..., "min": ..., "max": ..., "mean": ...}`.
</details>

In [ ]:
# ✏️ [EX5]


### Exercise 6: Stats for All Sensors

Use your `compute_stats()` function to compute statistics for **all three sensors** (`temp`, `humidity`, `pressure`). Store the results in a dictionary called `all_stats`.

**Expected output:**
```
Sensor Statistics
=================

temp:
  Readings: 5
  Min: 22.20, Max: 24.00, Mean: 23.12

humidity:
  Readings: 4
  Min: 43.50, Max: 47.30, Mean: 44.90

pressure:
  Readings: 4
  Min: 1011.90, Max: 1013.50, Mean: 1012.86
```

<details><summary>💡 Hint</summary>

Create a list of sensor names: `["temp", "humidity", "pressure"]`. Loop through it, call `compute_stats()` for each, and store in `all_stats[sensor_name] = stats`.
</details>

In [ ]:
# ✏️ [EX6]


---
## Part 7: Step 5 — Generate Report

Now let's build a nicely formatted report string that summarizes everything.

### Exercise 7: Generate Summary Report

Write a function called `generate_report(all_stats, total_rows, clean_count, bad_count)` that returns a formatted string containing the full summary report.

The report should look like this:
```
======================================
    SENSOR LOG SUMMARY REPORT
======================================

DATA OVERVIEW
-------------
Total readings:   20
Valid readings:   15
Invalid readings: 5
Data quality:     75.0%

SENSOR: temp
-------------
  Readings: 5
  Min:      22.20
  Max:      24.00
  Mean:     23.12

SENSOR: humidity
-------------
  Readings: 4
  Min:      43.50
  Max:      47.30
  Mean:     44.90

SENSOR: pressure
-------------
  Readings: 4
  Min:      1011.90
  Max:      1013.50
  Mean:     1012.86

======================================
```

<details><summary>💡 Hint</summary>

Build the report string piece by piece using `+=` or a list of lines that you `"\n".join()` at the end. Use f-strings for formatting numbers.
</details>

In [ ]:
# ✏️ [EX7]


---
## Part 8: Step 6 — Write Clean Data

Now let's write the cleaned data to a new CSV file.

### Exercise 8: Write Clean CSV

Write the `clean_data` to a file called `"sensor_data_clean.csv"`. Include the header row. Then read the file back and print the first 5 lines to verify.

**Expected output:**
```
Clean data written to sensor_data_clean.csv (15 rows + header)

First 5 lines of clean file:
  timestamp,sensor,value
  2024-01-15 08:00,temp,22.5
  2024-01-15 08:05,humidity,45.2
  2024-01-15 08:10,temp,23.1
  2024-01-15 08:15,pressure,1013.25
```

<details><summary>💡 Hint</summary>

Open the file in write mode. Write the header `"timestamp,sensor,value\n"` first. Then loop through `clean_data` and write each row joined by commas.
</details>

In [ ]:
# ✏️ [EX8]


---
## Part 9: Step 7 — Write Summary Report

Now write the summary report to a text file.

### Exercise 9: Write Report to File

Write the report string (from Exercise 7) to a file called `"sensor_report.txt"`. Print a confirmation and then read back the file to verify.

**Expected output:**
```
Report written to sensor_report.txt

--- sensor_report.txt ---
======================================
    SENSOR LOG SUMMARY REPORT
======================================
...
```

<details><summary>💡 Hint</summary>

Simply open `"sensor_report.txt"` in write mode and call `file.write(report)` with the report string you generated in Exercise 7.
</details>

In [ ]:
# ✏️ [EX9]


---
## Part 10: Putting It All Together

Now it's time to combine everything into a single, complete program.

### Exercise 10: The Complete Program

Write a `main()` function that does everything:
1. Reads `sensor_data.csv`
2. Validates and cleans the data
3. Computes statistics for each sensor
4. Generates a summary report
5. Writes `sensor_data_clean.csv`
6. Writes `sensor_report.txt`
7. Prints a final status message

Combine all your functions from previous exercises. Call `main()` at the end.

**Expected output:**
```
Sensor Log Summary Tool
=======================

[1/6] Reading sensor_data.csv...
      Found 20 data rows.

[2/6] Cleaning data...
      Valid: 15 | Invalid: 5

[3/6] Computing statistics...
      Processed 3 sensors.

[4/6] Generating report...
      Report ready.

[5/6] Writing sensor_data_clean.csv...
      Done.

[6/6] Writing sensor_report.txt...
      Done.

All tasks completed successfully!
```

<details><summary>💡 Hint</summary>

Copy your functions (`validate_row`, `compute_stats`, `generate_report`) into this cell. Then write a `main()` function that calls them in sequence. Use `print()` statements to show progress.
</details>

In [ ]:
# ✏️ [EX10]


---
## Bonus Exercises

These are optional challenges for students who finish early.

### Exercise 11: BONUS — Date/Time Filtering (Challenge)

Add a feature to your program that filters data by a time range. Write a function called `filter_by_time(data, start_time, end_time)` that returns only rows where the timestamp is between `start_time` and `end_time` (inclusive).

Since timestamps are in `YYYY-MM-DD HH:MM` format, you can compare them as strings (alphabetical comparison works correctly for this format).

**Expected output:**
```
Filtering: 2024-01-15 08:30 to 2024-01-15 09:00
Found 7 readings in time range.

Filtered data:
  2024-01-15 08:30 | temp      | 22.8
  2024-01-15 08:35 | humidity  | 46.1
  ...
```

<details><summary>💡 Hint</summary>

String comparison works for ISO-format dates: `"2024-01-15 08:30" >= start_time and "2024-01-15 08:30" <= end_time`. Filter the list using this condition on `row[0]`.
</details>

In [ ]:
# ✏️ [EX11]


### Exercise 12: BONUS — Sensor Comparison (Challenge)

Write a function called `compare_sensors(all_stats)` that prints a comparison table showing all sensors side by side.

**Expected output:**
```
Sensor Comparison Table
=======================

Metric       temp        humidity    pressure   
------       ----        --------    --------   
Count        5           4           4          
Min          22.20       43.50       1011.90    
Max          24.00       47.30       1013.50    
Mean         23.12       44.90       1012.86    
Range        1.80        3.80        1.60       
```

The **range** is `max - min`.

<details><summary>💡 Hint</summary>

Use f-string formatting with fixed widths (e.g., `f"{value:<12}"`) to align columns. Loop through the metrics (count, min, max, mean) and for each metric, print the value for each sensor.
</details>

In [ ]:
# ✏️ [EX12]


---
## What's Next?

Congratulations on completing **Computer Programming I**! You have come a long way from your first `print("Hello, World!")` to building a complete data processing tool.

Here is a summary of what you learned this semester:

| Week | Topic |
|:---|:---|
| 1 | Variables, Types & print() |
| 2 | Operators, f-strings & Type Conversion |
| 3 | Conditionals (if/elif/else) |
| 4 | for Loops & range() |
| 5 | while Loops, break & continue |
| 6 | Problem-Solving Patterns |
| 7 | Lists Fundamentals |
| 8 | 2D Lists & Nested Loops |
| 9 | String Processing |
| 10 | Functions: Basics |
| 11 | Scope & Mini-Library |
| 12 | Error Handling |
| 13 | File I/O & CSV |
| 14 | Mini Project: Sensor Log Summary |

### In Computer Programming II, you will explore:

- **Dictionaries and Sets** — more powerful data structures
- **Object-Oriented Programming (OOP)** — classes, objects, inheritance
- **Error Handling** — try/except, custom exceptions
- **Modules and Libraries** — NumPy, Pandas, Matplotlib
- **Working with APIs** — fetching data from the internet
- **Databases** — storing data with SQLite
- **Larger Projects** — putting it all together

Keep coding and keep learning. The best way to improve is to **practice every day**, even if it's just for 15 minutes.

> 💡 **Note:** If you enjoyed this course, consider exploring Python projects on your own: build a simple game, automate a daily task, or analyze data that interests you!